# LoRA / QLoRA Fine-tuning

This notebook shows a PEFT + TRL fine-tuning pipeline with a small instruction dataset, trainable-parameter reporting, adapter saving, and a before/after generation check.

In [ ]:
from pathlib import Path
import os
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'llm_lab').exists():
        sys.path.insert(0, str(candidate))
        repo_root = candidate
        break

from llm_lab.demo_data import SAMPLE_QA
from llm_lab.models import count_trainable_parameters, load_causal_lm

base_model_name = os.environ.get('LLM_LAB_FINETUNE_MODEL', 'sshleifer/tiny-gpt2')
print(f'Repo root: {repo_root}')
print(f'Base model: {base_model_name}')
print('Sample rows:', SAMPLE_QA[:2])

In [ ]:
bundle = load_causal_lm(base_model_name, quantized=False, device_map=None)
base_model = bundle.model
tokenizer = bundle.tokenizer
try:
    from peft import LoraConfig, get_peft_model
    lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=['c_attn', 'q_proj', 'v_proj'], lora_dropout=0.05, bias='none', task_type='CAUSAL_LM')
    lora_model = get_peft_model(base_model, lora_config)
    parameter_report = count_trainable_parameters(lora_model)
except Exception as exc:
    parameter_report = {'status': 'PEFT unavailable', 'reason': str(exc)}
parameter_report

In [ ]:
train_dataset = [
    {"prompt": f"### Instruction:\n{row['question']}\n### Response:\n", "response": row['answer']}
    for row in SAMPLE_QA
]
train_dataset

In [ ]:
training_status = {'status': 'not run'}
try:
    from datasets import Dataset
    from transformers import TrainingArguments
    from trl import SFTTrainer

    dataset = Dataset.from_list([{'text': f"{row['prompt']}{row['response']}"} for row in train_dataset])
    training_args = TrainingArguments(output_dir=str(repo_root / 'outputs' / 'qlora'), num_train_epochs=1, per_device_train_batch_size=1, gradient_accumulation_steps=1, logging_steps=1, save_steps=5, report_to='none', learning_rate=2e-4)
    trainer = SFTTrainer(model=lora_model if 'lora_model' in globals() else base_model, train_dataset=dataset, dataset_text_field='text', args=training_args, tokenizer=tokenizer, max_seq_length=256)
    trainer.train()
    adapter_dir = repo_root / 'outputs' / 'lora_adapter'
    trainer.model.save_pretrained(adapter_dir)
    training_status = {'status': 'trained', 'adapter_dir': str(adapter_dir), 'logs': trainer.state.log_history}
except Exception as exc:
    training_status = {'status': 'skipped', 'reason': str(exc)}
training_status

In [ ]:
before_after = {'status': 'not run'}
try:
    prompt = 'Write two bullet points explaining why LoRA is efficient.'
    inputs = tokenizer(prompt, return_tensors='pt')
    base_text = base_model.generate(**inputs, max_new_tokens=40, do_sample=False)
    before_after = {'baseline': tokenizer.decode(base_text[0], skip_special_tokens=True)}
    if 'lora_model' in globals():
        tuned_text = lora_model.generate(**inputs, max_new_tokens=40, do_sample=False)
        before_after['tuned'] = tokenizer.decode(tuned_text[0], skip_special_tokens=True)
except Exception as exc:
    before_after = {'status': 'failed', 'reason': str(exc)}
before_after

The notebook is intentionally tiny by default. On a GPU machine, swap in a larger base model and a public instruction dataset to reproduce the full QLoRA flow with a real loss curve and adapter checkpoint.